In [1]:
import sys
sys.path.append("..")  

In [2]:
import pandas as pd

from src.evaluation import categorical_scores

samples = pd.read_parquet("../data/samples.parquet")

per_day = samples.groupby("day").agg(
    samples=("label", "size"),
    times=("time", "nunique"),
    positive_rate=("label", "mean"),
    persistence_positive_rate=("flash_now", "mean"),
)
print(per_day)

print("\nPersistence scores (flash at t -> flash at t + 30 min):")
for day, day_samples in samples.groupby("day"):
    scores = categorical_scores(day_samples["label"], day_samples["flash_now"])
    print(day, {name: round(value, 3) for name, value in scores.items()})

overall_scores = categorical_scores(samples["label"], samples["flash_now"])
print("all days", {name: round(value, 3) for name, value in overall_scores.items()})
new_flash_share = overall_scores["misses"] / (overall_scores["hits"] + overall_scores["misses"])
print(f"share of positive cells with no flash 30 min earlier: {new_flash_share:.2f}")

            samples  times  positive_rate  persistence_positive_rate
day                                                                 
2026-07-18   325804     47       0.011271                   0.011031
2026-07-19   277280     40       0.005366                   0.005586
2026-07-20   325804     47       0.007557                   0.007535
2026-07-21   311940     45       0.021350                   0.020927
2026-07-22   325804     47       0.000414                   0.000718

Persistence scores (flash at t -> flash at t + 30 min):
2026-07-18 {'hits': 1886, 'misses': 1786, 'false_alarms': 1708, 'pod': 0.514, 'far': 0.475, 'csi': 0.351}
2026-07-19 {'hits': 622, 'misses': 866, 'false_alarms': 927, 'pod': 0.418, 'far': 0.598, 'csi': 0.258}
2026-07-20 {'hits': 1074, 'misses': 1388, 'false_alarms': 1381, 'pod': 0.436, 'far': 0.563, 'csi': 0.279}
2026-07-21 {'hits': 2483, 'misses': 4177, 'false_alarms': 4045, 'pod': 0.373, 'far': 0.62, 'csi': 0.232}
2026-07-22 {'hits': 32, 'misses': 103, '